# Pseudospectra of a non-normal operator

The eigenvalues of an operator tell you how perturbations behave *eventually*.
They do not tell you what happens on the way there. When an operator is
**non-normal** — its eigenvectors are not orthogonal — a disturbance can grow by
orders of magnitude before the modal decay eventually wins.

The $\varepsilon$-pseudospectrum makes that visible. It is the set of $z$ where

$$\sigma_{\min}(zI - A) \le \varepsilon,$$

equivalently where the resolvent norm $\|(zI-A)^{-1}\|$ exceeds $1/\varepsilon$.
For a *normal* operator this collapses onto disks of radius $\varepsilon$ around
the eigenvalues. For a non-normal one it bulges far beyond them, and the size of
that bulge is what nonmodal stability analysis measures.

This notebook works through one operator end to end using matrices from the
[NIST Matrix Market](https://math.nist.gov/MatrixMarket/).

In [ ]:
import pathlib
import sys

# Reuse the test suite's Matrix Market fetcher: downloads are checksum-verified
# and cached under tests/_data, so re-running this notebook costs nothing.
sys.path.insert(0, str(pathlib.Path.cwd().parent / 'tests'))

import matplotlib.pyplot as plt
import numpy as np
import scipy.linalg
from matrixmarket import MATRICES, load_dense

from nonmodal import Bounds, choose_contour_levels, sample_sigmin, uniform_points

## The operator

`west0479` is a 479x479 Jacobian from a chemical-engineering model, and a
standard example in the pseudospectra literature. Its *normality defect*
$\|AA^* - A^*A\| / \|A\|^2$ is far from zero, so we should expect the
pseudospectrum to depart substantially from the eigenvalues.

In [ ]:
spec = MATRICES['west0479']
A = load_dense(spec)
eigvals = np.linalg.eigvals(A)

commutator = A @ A.conj().T - A.conj().T @ A
defect = np.linalg.norm(commutator) / np.linalg.norm(A) ** 2

print(f'{spec.name}: {A.shape[0]}x{A.shape[1]}')
print(f'normality defect      : {defect:.3f}   (0 would mean normal)')
print(f'rightmost eigenvalue  : {eigvals.real.max():.4g}')

## Sampling

Sampling needs the Schur factor, not the operator itself: $zI - T$ is triangular,
so each $\sigma_{\min}$ costs a pair of triangular solves rather than a full
factorisation. `sample_sigmin` is the primitive — it evaluates a flat array of
complex points, and everything else in the library is a question of *which*
points you hand it.

In [ ]:
T = np.asarray(scipy.linalg.schur(A, output='complex')[0], dtype=np.complex128)

bounds = Bounds.around_spectrum(eigvals, pad=0.25)
z = uniform_points(bounds, nx=45, ny=45)
sigmin = sample_sigmin(z, T, nprocs=4)

print(f'{z.size} points, sigma_min spans '
      f'{sigmin.min():.2e} .. {sigmin.max():.2e} '
      f'({np.log10(sigmin.max() / sigmin.min()):.1f} decades)')

## The picture

Contours are drawn straight from the sampled points via a Delaunay
triangulation — the same thing `nonmodal plot` does — so no interpolation is
involved. Black dots are the eigenvalues.

Note how far the outer contours extend beyond the spectrum. That gap *is* the
non-normality.

In [ ]:
levels = choose_contour_levels(sigmin, min_level=1e-6, nlevels=8)

fig, ax = plt.subplots(figsize=(7.5, 6))
cs = ax.tricontour(z.real, z.imag, np.log10(sigmin),
                   levels=np.log10(levels), cmap='turbo')
ax.clabel(cs, fmt=lambda v: f'$10^{{{v:.0f}}}$', fontsize=8)
ax.scatter(eigvals.real, eigvals.imag, s=4, c='k', label='eigenvalues')

ax.set_xlabel(r'$\mathrm{Re}\,z$')
ax.set_ylabel(r'$\mathrm{Im}\,z$')
ax.set_title(r'$\varepsilon$-pseudospectrum of west0479')
ax.legend(loc='upper left')
plt.show()

## How much does non-normality buy?

For *any* matrix, $\sigma_{\min}(zI-A) \le \mathrm{dist}(z, \Lambda(A))$, with
equality exactly when the matrix is normal. So the ratio

$$\frac{\sigma_{\min}(zI-A)}{\mathrm{dist}(z,\Lambda)}$$

is 1 everywhere for a normal operator, and drops below 1 by however much the
non-normality amplifies the resolvent. Its reciprocal is the factor by which a
purely modal argument would *understate* the response.

In [ ]:
def spectral_distance(points, eigenvalues):
    """dist(z, spectrum) for each sample point."""
    return np.min(np.abs(points[:, None] - eigenvalues[None, :]), axis=1)


ratio = sigmin / spectral_distance(z, eigvals)
print(f'sigma_min / dist(z, spectrum):  min {ratio.min():.2e}, max {ratio.max():.3f}')
print(f'peak resolvent amplification vs the modal estimate: {1 / ratio.min():.0f}x')

## Contrast: a normal operator

`bcsstk01` is a symmetric stiffness matrix, hence normal. The same ratio should
be 1 to within solver tolerance — the pseudospectrum really is just disks around
the eigenvalues, and there is nothing nonmodal to find.

In [ ]:
normal_spec = MATRICES['bcsstk01']
B = load_dense(normal_spec)
b_eigvals = np.linalg.eigvals(B)
B_T = np.asarray(scipy.linalg.schur(B, output='complex')[0], dtype=np.complex128)

b_bounds = Bounds.around_spectrum(b_eigvals, pad=0.25)
b_z = uniform_points(b_bounds, nx=20, ny=20)
b_sigmin = sample_sigmin(b_z, B_T, nprocs=4)

b_ratio = b_sigmin / spectral_distance(b_z, b_eigvals)
print(f'{normal_spec.name} (symmetric, so normal)')
print(f'  ratio range: {b_ratio.min():.6f} .. {b_ratio.max():.6f}')
print(f'  max deviation from 1: {np.abs(b_ratio - 1).max():.2e}')

## Writing the interactive version

`pseudo_contours` produces the Plotly page that `nonmodal plot` emits. Pass
`inline_js=True` to embed plotly.js so the file also renders on a machine
without network access (at the cost of a few MB).

In [ ]:
from nonmodal import pseudo_contours

out = pseudo_contours('.', 'west0479_contours.html', z, sigmin, eigvals, levels)
print(f'wrote {out} ({pathlib.Path(out).stat().st_size / 1024:.0f} KB)')

## Where to go next

- `sample_sigmin` takes any point set, so you are not restricted to a lattice.
  `nonmodal.refine` grows one adaptively, concentrating samples where a linear
  interpolant of $\log_{10}\sigma_{\min}$ is worst.
- On the command line the same workflow is
  `nonmodal run` followed by `nonmodal plot`; see the project README.
- Other matrices in `MATRICES` span the range from normal to strongly
  non-normal: `bcsstk01` (0.0), `gre__115` (0.03), `rw136` (0.05),
  `olm100` (0.28), `west0479` (0.63).